# Lesson 5.1: Confidence Intervals

Build a 95% z-interval for a mean when sigma is known, then simulate thousands of intervals to see about 95% catch the true mean.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A standardised exam is known to have an SD of 12 points (sigma = 12). A random sample of 40 students averaged 78.5. The question: what is the true average for ALL students? A single number hides the uncertainty, so we build a 95% confidence interval and then test whether such intervals really catch the truth.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: The z-interval by formula

The lesson's exam: sigma = 12 is known, n = 40 and the sample mean is 78.5. Three steps: standard error, margin of error (z* times SE), then the interval.

In [1]:
import numpy as np  # Load NumPy for the square root and arrays.
from scipy import stats  # Load SciPy's stats tools, which hold the normal distribution.

sigma, n, xbar = 12, 40, 78.5  # Store the three facts we know: sigma (known SD of scores), n (sample size) and xbar (sample mean).
z = stats.norm.ppf(0.975)  # Find z*, the number of standard errors that captures the middle 95%: ppf(0.975) leaves 2.5% in each tail.
se = sigma / np.sqrt(n)  # Standard error of the mean: sigma divided by the square root of n.
me = z * se  # Margin of error = z* times SE, the 'plus or minus' of the interval.

print(f'z* = {z:.3f}, SE = {se:.3f}, margin = {me:.2f}')  # Print z*, the SE and the margin, rounded.
print(f'95% CI = ({xbar - me:.2f}, {xbar + me:.2f})')  # Print the interval: the estimate minus and plus the margin.

z* = 1.960, SE = 1.897, margin = 3.72
95% CI = (74.78, 82.22)


**What this means.** z* is 1.960, the standard error is 1.897 and the margin of error is 3.72 points. So the 95% interval is 74.78 to 82.22. We are 95% confident the true average for all students lies in that range, not that 95% of students score in it.

## Step 2: Confidence level changes the width

Being more confident means casting a wider net. We repeat the calculation for 90%, 95% and 99%, using the matching z*.

In [2]:
for level in (0.90, 0.95, 0.99):  # Loop over three confidence levels.
    z = stats.norm.ppf(1 - (1 - level) / 2)  # Find z* for each level; 1 - (1 - level)/2 puts the leftover probability equally in both tails.
    me = z * se  # Compute the margin of error at that level, reusing the same standard error.
    print(f'{level:.0%}: z* = {z:.3f}, margin = {me:.2f}, ({xbar - me:.2f}, {xbar + me:.2f})')  # Print the level, z*, the margin and the interval.

90%: z* = 1.645, margin = 3.12, (75.38, 81.62)
95%: z* = 1.960, margin = 3.72, (74.78, 82.22)
99%: z* = 2.576, margin = 4.89, (73.61, 83.39)


**What this means.** More confidence needs a wider net: 90% gives a margin of 3.12 (75.38 to 81.62), 95% gives 3.72 (74.78 to 82.22) and 99% gives 4.89 (73.61 to 83.39). The price of being surer is a vaguer answer.

## Step 3: Sample size changes the width

At 95%, quadrupling n halves the margin of error, the 1 over square root of n rule again.

In [3]:
for n_try in (10, 40, 160, 640):  # Loop over four sample sizes, each four times the last.
    margin = 1.96 * sigma / np.sqrt(n_try)  # Margin of error at 95% for that n, using z* = 1.96.
    print(f'n = {n_try:3d}: margin = {margin:.2f}')  # Print n and the margin.

n =  10: margin = 7.44
n =  40: margin = 3.72
n = 160: margin = 1.86
n = 640: margin = 0.93


**What this means.** The margin falls from 7.44 at n = 10, to 3.72 at n = 40, 1.86 at n = 160 and 0.93 at n = 640. Each time n is quadrupled the margin halves, the 1 over square root of n rule again.

## Step 4: Does it really catch the truth?

Make the truth known: a normal population with mean 78.5 and SD 12. Draw 10,000 samples of 40, build a 95% interval from each, and count how many contain 78.5.

In [4]:
rng = np.random.default_rng(1)  # Make a seeded random generator so the check is repeatable.
true_mean = 78.5  # Pretend we know the truth: the real mean is 78.5.
means = rng.normal(true_mean, sigma, size=(10_000, 40)).mean(axis=1)  # Draw 10,000 samples of 40 scores from a normal population (mean 78.5, SD 12) and take each sample's mean.
me = 1.96 * sigma / np.sqrt(40)  # Margin of error for n = 40, as before.
caught = (means - me <= true_mean) & (true_mean <= means + me)  # For each sample mean, check whether the interval mean +/- margin contains the true mean.

print(f'intervals that caught the true mean: {caught.mean():.1%}')  # Print the share of intervals that caught the true mean.
print('intervals that missed:', (~caught).sum(), 'of 10000')  # Print how many missed.

intervals that caught the true mean: 95.0%
intervals that missed: 503 of 10000


**What this means.** Of 10,000 intervals, 95.0% caught the true mean and 503 missed. That is exactly what '95% confident' means: the METHOD captures the truth about 95 times in 100, though any single interval either does or does not.

## So, what is the answer?

With 40 students averaging 78.5 and sigma = 12, the 95% interval for the true average is 74.78 to 82.22. The simulation shows the method works: 95.0% of intervals caught the truth. Caution: this recipe assumes sigma is truly known and the sample is random; if sigma is estimated from the data, use the t-interval of the next lesson.

## Now you try

A packing machine fills bags of rice. Its fill weight is known to vary with an SD of 0.5 kg (sigma = 0.5). An inspector weighs 64 random bags and finds an average of 12.3 kg. She wants an interval for the machine's true average fill and to know how large a sample would shrink the margin to 0.05 kg.

**Data:** sigma = 0.5, n = 64, sample mean = 12.3 kg.

1. Build the 95% interval for the true mean fill.
2. Build the 99% interval and compare its width.
3. How many bags would give a 95% margin of 0.05 kg?

*Hint:* Use xbar +/- z* * sigma/sqrt(n). For the sample size, rearrange margin = 1.96 * sigma / sqrt(n) to n = (1.96 * sigma / margin)^2 and round up.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np
from scipy import stats
sigma, n, xbar = 0.5, 64, 12.3
se = sigma / np.sqrt(n)
for level in (0.95, 0.99):
    z = stats.norm.ppf(1 - (1 - level) / 2)
    print(f'{level:.0%}: z* = {z:.3f}, margin = {z * se:.3f}, ({xbar - z * se:.2f}, {xbar + z * se:.2f})')
need = int(np.ceil((1.96 * sigma / 0.05) ** 2))
print('n for a 95% margin of 0.05:', need)

95%: z* = 1.960, margin = 0.122, (12.18, 12.42)
99%: z* = 2.576, margin = 0.161, (12.14, 12.46)
n for a 95% margin of 0.05: 385


**Answer.** The 95% interval is 12.18 to 12.42 kg (margin 0.122) and the 99% interval is 12.14 to 12.46 (margin 0.161). The 99% interval is wider: being surer costs precision. To cut the 95% margin to 0.05 kg she would need 385 bags.

## The same question in R, SPSS and Stata

**R**

```r
sigma <- 12; n <- 40; xbar <- 78.5  # Store sigma, n and the sample mean on one line using semicolons.
z <- qnorm(0.975)  # z* from the normal distribution: qnorm(0.975).
me <- z * sigma / sqrt(n)  # Margin of error = z* times sigma / sqrt(n).
c(z, me, xbar - me, xbar + me)  # Print z*, the margin and the interval limits in one vector.
set.seed(1)  # Fix the random seed.
means <- replicate(10000, mean(rnorm(40, 78.5, 12)))  # Simulate 10,000 sample means of 40 scores from a normal population (78.5, 12).
mean(abs(means - 78.5) <= me)  # Print the share of sample means within one margin of the truth; this equals the share of intervals that catch it.
```

How to read it: The c(...) line prints 1.960, 3.72, 74.78 and 82.22, matching Python. The last line should print close to 0.95: the coverage, simulated so close to but not identical to Python's 95.0%.

**SPSS** (Transform > Compute Variable (use IDF.NORMAL in the function list))

```
DATA LIST FREE / sigma n xbar.
BEGIN DATA
12 40 78.5
END DATA.
COMPUTE z = IDF.NORMAL(0.975, 0, 1).
COMPUTE se = sigma / SQRT(n).
COMPUTE me = z * se.
COMPUTE lower = xbar - me.
COMPUTE upper = xbar + me.
LIST VARIABLES=z se me lower upper.
```

How to read it: The listing shows z about 1.960, se 1.897, me 3.72, lower 74.78 and upper 82.22, the same as Python.

**Stata**

```
scalar z = invnormal(0.975)
scalar se = 12 / sqrt(40)
scalar me = z * se
display "z* = " z
display "SE = " se
display "margin = " me
display "95% CI = (" (78.5 - me) ", " (78.5 + me) ")"
```

How to read it: The five display lines show z* = 1.96, SE = 1.897, margin = 3.72 and the interval (74.78, 82.22). These match Python exactly because no randomness is involved.